# arXiv Tools

![arXiv Tools](https://proto-bio.github.io/proto-assets/images/tool/arxiv/hero.png)

This notebook demonstrates `run_arxiv_search`, which searches arXiv by title, author, abstract, or category. Each hit carries its versioned arXiv ID, title, authors, abstract, submission dates, categories, DOI and journal reference when present, and abstract and PDF links.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("arxiv")
display_overview("arxiv")
display_docs_section("arxiv", "Background")

# arXiv

[arXiv](https://arxiv.org/) is the open preprint server run by Cornell University for physics, mathematics, computer science, quantitative biology, and related fields. This toolkit exposes one tool, `arxiv-search`, which queries the arXiv API by title, author, abstract, or category and returns each preprint's versioned ID, title, authors, abstract, dates, categories, DOI and journal reference when present, and abstract and PDF links. It runs on CPU in process and needs only network access.

[arXiv](https://arxiv.org/) has distributed preprints since 1991 and now holds more than two million. Machine learning methods for biology, including many protein structure prediction and design models, often appear there first, in categories such as `q-bio.BM` (biomolecules), `q-bio.QM` (quantitative methods), and `cs.LG` (machine learning). Submissions are moderated but not peer reviewed. Each has an identifier such as `2504.00146`, and each revision adds a version suffix (`v1`, `v2`), so a citation can name the exact text that was read.

## Available tools

In [2]:
display_available_tools("arxiv")

- **`run_arxiv_search()`** — Search arXiv preprints (q-bio, cs.LG and other categories) by title, author, abstract or category for IDs, abstracts, DOIs and PDF links

### `run_arxiv_search`

`run_arxiv_search` takes arXiv `search_query` strings built from field prefixes such as `ti:`, `au:`, `abs:`, and `cat:`. Here we search titles for protein binder design.

In [3]:
from proto_tools import (
    ArxivSearchConfig,
    ArxivSearchInput,
    run_arxiv_search,
)

In [4]:
# Display docs
display_api_reference("arxiv", "input", "run_arxiv_search")

# Search titles for a phrase
inputs = ArxivSearchInput(queries='ti:"protein binder"')

**Input** — `ArxivSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>queries</code> | <code>list[str]</code> | required | arXiv search_query strings, e.g. 'ti:"protein binder" AND cat:q-bio.BM' |

In [5]:
# Display config docs
display_api_reference("arxiv", "config", "run_arxiv_search")

# Return the 5 most relevant hits | see docs above for all fields
config = ArxivSearchConfig(max_results=5)

**Config** — `ArxivSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>max_results</code> | <code>int</code> | <code>10</code> | Maximum hits returned per query |
| <code>sort_by</code> | <code>Literal['relevance', 'submittedDate', 'lastUpdatedDate']</code> | <code>'relevance'</code> | relevance, or newest first by submittedDate or lastUpdatedDate |

In [6]:
# Run the search
output = run_arxiv_search(inputs, config)

In [7]:
# Display output docs
display_api_reference("arxiv", "output", "run_arxiv_search")

# Inspect the ranked hits
result = output.results[0]
print(f"{result.total_count} matching preprints")
for hit in result.hits:
    print(f"{hit.arxiv_id}  {hit.published[:10]}  [{hit.primary_category}]  {hit.title[:70]}")

**Output** — `ArxivSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[ArxivSearchResult]</code> | <code>[]</code> | Results in input order |
| <code>source_url</code> | <code>str</code> | required | arXiv API endpoint used |

**`ArxivSearchResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query</code> | <code>str</code> | required | Query as sent |
| <code>total_count</code> | <code>int</code> | required | Total matching preprints |
| <code>hits</code> | <code>list[ArxivSearchHit]</code> | <code>[]</code> | Hits in the requested order |

**`ArxivSearchHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>arxiv_id</code> | <code>str</code> | required | arXiv ID with version, e.g. '2504.00146v2' |
| <code>title</code> | <code>str</code> | required | Title |
| <code>authors</code> | <code>list[str]</code> | <code>[]</code> | Author names |
| <code>summary</code> | <code>str</code> | required | Abstract |
| <code>published</code> | <code>str</code> | required | Submission time of version 1 (ISO 8601) |
| <code>updated</code> | <code>str</code> | required | Submission time of this version (ISO 8601) |
| <code>primary_category</code> | <code>str &#124; None</code> | <code>None</code> | Primary category |
| <code>categories</code> | <code>list[str]</code> | <code>[]</code> | All arXiv categories |
| <code>doi</code> | <code>str &#124; None</code> | <code>None</code> | DOI of the published version, if given |
| <code>journal_ref</code> | <code>str &#124; None</code> | <code>None</code> | Journal reference, if given |
| <code>comment</code> | <code>str &#124; None</code> | <code>None</code> | Author comment (pages, venue) |
| <code>abs_url</code> | <code>str</code> | required | arXiv abstract page |
| <code>pdf_url</code> | <code>str</code> | required | arXiv PDF |

15 matching preprints
2504.00146v2  2025-03-31  [cs.LG]  Why risk matters for protein binder design
2605.04118v2  2026-05-05  [q-bio.QM]  ProtDBench: A Unified Benchmark of Protein Binder Design and Evaluatio
2608.20755v2  2026-08-21  [cs.AI]  Natural-Language-Guided Generator-Agnostic Shortlisting for Protein Bi
2405.01616v1  2024-05-02  [q-bio.BM]  Generative Active Learning for the Search of Small-molecule Protein Bi
2611.00281v1  2026-10-30  [cs.LG]  DELBERT-2: Pretrained Fingerprint Language Models for DEL Protein Bind


#### Combining Fields and Sorting by Date

Field prefixes combine with `AND`, `OR`, and `ANDNOT`. Here we run two queries in one call, keeping biomolecule preprints that mention diffusion in the abstract and recent protein language model preprints, newest first. arXiv asks for at most one request every three seconds, so the tool waits between the two queries.

In [8]:
recent_output = run_arxiv_search(
    ArxivSearchInput(queries=["cat:q-bio.BM AND abs:diffusion AND abs:protein", 'abs:"protein language model"']),
    ArxivSearchConfig(max_results=3, sort_by="submittedDate"),
)

for result in recent_output.results:
    print(f"\n{result.query}")
    for hit in result.hits:
        print(f"  {hit.arxiv_id}  {hit.published[:10]}  {hit.title[:70]}")
        print(f"    {hit.abs_url}  doi={hit.doi}")


cat:q-bio.BM AND abs:diffusion AND abs:protein
  2611.03963v1  2026-11-02  ProteoCraft: Protein Design as Proposal-Verification Search
    https://arxiv.org/abs/2611.03963v1  doi=None
  2610.37410v1  2026-10-27  Accurate structural modelling of protein-glycan recognition with Glyco
    https://arxiv.org/abs/2610.37410v1  doi=None
  2610.33195v1  2026-10-23  Agent-guided epitope excision enables economical de novo binder design
    https://arxiv.org/abs/2610.33195v1  doi=None

abs:"protein language model"
  2611.05055v1  2026-11-03  Ablating Design Choices for Protein Ensemble Generators Across Structu
    https://arxiv.org/abs/2611.05055v1  doi=None
  2610.26747v1  2026-10-19  recAMPLIFY: Large-Scale Pretraining of Recursive Protein Language Mode
    https://arxiv.org/abs/2610.26747v1  doi=None
  2610.26651v1  2026-10-19  The Effectiveness and Durability of Pretraining Data Filtering in Prot
    https://arxiv.org/abs/2610.26651v1  doi=None


## Export Results

Search results can be saved to JSON or CSV. arXiv metadata is CC0, so it can be stored and shared freely; this example still writes to a temporary directory to keep the repository clean.

In [9]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as output_dir:
    output.export("arxiv_search", export_path=output_dir, file_format="csv")
    print(f"Exported {sorted(path.name for path in Path(output_dir).iterdir())}")

Exported ['arxiv_search.csv']
